In [3]:
!pip install -q faiss-cpu

In [4]:
from google import genai
from google.colab import userdata

import numpy as np
import faiss
import os
import json

from google.colab import drive
drive.mount('/content/drive')

# Gemini client
client = genai.Client(
    api_key=userdata.get("Assign_2_API")
)

folder_path = "/content/drive/MyDrive/RAG_Assignment"

docs = [
    "Makeup.txt",
    "healthy_lifestyle.txt",
    "saudi_cities.txt"
]

for document in docs:
    file_path = os.path.join(folder_path, document)
    print(document, "->", os.path.exists(file_path))

documents = {}

for document in docs:
    file_path = os.path.join(folder_path, document)

    with open(file_path, "r", encoding="utf-8") as file:
        documents[document] = file.read()

for document, text in documents.items():
    print(document)
    print("Number of characters:", len(text))
    print("-" * 50)

Mounted at /content/drive
Makeup.txt -> True
healthy_lifestyle.txt -> True
saudi_cities.txt -> True
Makeup.txt
Number of characters: 3466
--------------------------------------------------
healthy_lifestyle.txt
Number of characters: 3955
--------------------------------------------------
saudi_cities.txt
Number of characters: 2074
--------------------------------------------------


In [5]:
def create_dynamic_chunks(text, max_chars=1000):

    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n")
        if paragraph.strip()
    ]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) + 1 <= max_chars:

            if current_chunk:
                current_chunk += "\n\n"

            current_chunk += paragraph

        else:

            if current_chunk:
                chunks.append(current_chunk)

            current_chunk = paragraph

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

#all chuncks for all docs
all_chunks = []

for document, text in documents.items():

    chunks = create_dynamic_chunks(
        text,
        max_chars=1000
    )

    for i, chunk in enumerate(chunks, start=1):

        all_chunks.append({
            "document": document,
            "chunk_index": i,
            "chunk_text": chunk
        })


print("Total chunks:", len(all_chunks))


# Display chunks
for item in all_chunks:
    print("File:", item["document"])
    print("Chunk:", item["chunk_index"])
    print(item["chunk_text"][:200])
    print("-" * 50)

Total chunks: 13
File: Makeup.txt
Chunk: 1
Makeup is a popular part of the beauty industry, and many brands offer different products for different needs. Common makeup products include foundation, concealer, lipstick, mascara, blush, and eyesh
--------------------------------------------------
File: Makeup.txt
Chunk: 2
Maybelline is a makeup brand that offers affordable beauty products. Its product range includes mascara, foundation, concealer, lipstick, and eyeliner. Maybelline products are available in many stores
--------------------------------------------------
File: Makeup.txt
Chunk: 3
Rare Beauty is a makeup brand founded by Selena Gomez. The brand offers products such as liquid blush, foundation, concealer, highlighter, and lip products. Rare Beauty is especially known for its liq
--------------------------------------------------
File: Makeup.txt
Chunk: 4
Charlotte Tilbury is a beauty brand known for its luxury makeup products. Its product range includes foundation, lipstick, 

In [6]:
def create_embedding(text):

    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )

    return response.embeddings[0].values

embeddings = []

for item in all_chunks:

    vector = create_embedding(item["chunk_text"])

    embeddings.append(vector)

print("Number of embeddings:", len(embeddings))

Number of embeddings: 13


In [7]:
embeddings = np.array(embeddings, dtype="float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (13, 3072)


In [8]:
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vector dimension:", dimension)
print("Number of vectors in FAISS:", index.ntotal)

faiss.write_index(index, "assignment_vectors.faiss")

print("FAISS database saved successfully.")

Vector dimension: 3072
Number of vectors in FAISS: 13
FAISS database saved successfully.


In [9]:
# Save chunk metadata
with open("chunk_metadata.json", "w", encoding="utf-8") as file:
    json.dump(all_chunks, file, ensure_ascii=False, indent=2)

print("Chunk metadata saved successfully.")

Chunk metadata saved successfully.


In [10]:
#phase 1 check
print("Files created:")

print("assignment_vectors.faiss ->",
      os.path.exists("assignment_vectors.faiss"))

print("chunk_metadata.json ->",
      os.path.exists("chunk_metadata.json"))

Files created:
assignment_vectors.faiss -> True
chunk_metadata.json -> True


In [11]:
#phase 2
# load the saved FAISS vector database
index = faiss.read_index("assignment_vectors.faiss")

# load the chunk metadata
with open("chunk_metadata.json", "r", encoding="utf-8") as file:
    all_chunks = json.load(file)

print("FAISS database loaded.")
print("Number of vectors:", index.ntotal)
print("Number of chunks:", len(all_chunks))

FAISS database loaded.
Number of vectors: 13
Number of chunks: 13


In [12]:
question = input("Enter your question: ")

print("\nQuestion:")
print(question)

question_embedding = create_embedding(question)

question_embedding = np.array(
    [question_embedding],
    dtype="float32"
)

faiss.normalize_L2(question_embedding)

print("Query embedding shape:", question_embedding.shape)


scores, indices = index.search(
    question_embedding,
    3
)

print("Similarity scores:")
print(scores)

print("Vector indices:")
print(indices)

Enter your question: What are the benefits of a healthy lifestyle?

Question:
What are the benefits of a healthy lifestyle?
Query embedding shape: (1, 3072)
Similarity scores:
[[0.7272571  0.72166455 0.69206095]]
Vector indices:
[[9 5 7]]


In [13]:
print("\nQuestion:")
print(question)

print("\nTop 3 Results:")
print("=" * 70)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    result = all_chunks[idx]

    print(f"\nRank: {rank}")
    print(f"Similarity Score: {score:.4f}")
    print(f"Reference File: {result['document']}")
    print(f"Chunk: {result['chunk_index']}")
    print("Answer:")
    print(result["chunk_text"])

    print("-" * 70)


Question:
What are the benefits of a healthy lifestyle?

Top 3 Results:

Rank: 1
Similarity Score: 0.7273
Reference File: healthy_lifestyle.txt
Chunk: 5
Answer:
Social connections can have a positive effect on overall well-being. Spending time with family, friends, and other people can provide emotional support and reduce feelings of loneliness. Healthy relationships can also encourage people to make better lifestyle choices. Activities such as walking with a friend, eating meals together, or participating in group activities can combine social interaction with healthy habits.

Building a healthy lifestyle takes time and consistency. People do not need to change all their habits at once. Small changes, such as drinking more water, walking regularly, sleeping earlier, or eating more vegetables, can gradually improve daily life. The most important part is to continue these habits over time. A healthy lifestyle is not about being perfect, but about making better choices consistently.
---